# `StruckeC14_Sweden_v1.csv` -> species + material transformation

Self-contained v1 pipeline: cleans up multi-valued id columns and `landskap` typos, then
splits/melts by species and material element, for the new dataset revision
`data/StruckeC14_Sweden_v1.csv`. Continuation in spirit of
`archive/notebooks/c14_dataset_tranformation.ipynb`'s steps 1-4, but this notebook never reads
anything from `output/` (old or new) - the SEAD taxonomy and material-element ids are recomputed
live against `sead_staging` every run, from the professionalized manual-resolution CSVs in
`data/manual_resolutions/` via the shared `shared.resolution.species`/`shared.resolution.material`
functions. Scope stays narrower than the old pipeline: this only melts by species and material
element - the measurement melt is out of scope here (see `archive/` for that).

Comparing `StruckeC14_Sweden_v1.csv` against the old `c14_master_v08.xlsx` showed it's the same
underlying 30,301 records, lightly re-exported:
- `material` vocabulary is unchanged (same 42 values, identical counts) - the professionalized
  `material_manual_resolution.csv` is the same hand-completed breakdown as before, joined
  case-insensitively since v1's text is lowercase where the resolution was built against Title Case.
- `species` vocabulary is ~97% unchanged; the ~3% that differs is punctuation normalization
  (`cf` -> `cf.`, trailing `.` added after `sp`) plus two real content changes: `naket korn`/
  `najet korn` (previously flagged uncertain as `naket?`) merged into an unambiguous `nakenkorn`,
  and a comma-less `en möjl. gran` variant. Reconciled below rather than re-running species
  resolution from scratch - `species_manual_resolution.csv` (the professionalized version of the
  hand-completed species mapping) is reused unchanged.
- a handful of records pack multiple `site_id`/`uppdragsnummer` values into a single comma-separated
  cell (e.g. `L2016:9874, L2015:343`) - not something the old pipeline had to deal with, so it's
  handled as its own step here (step 2) rather than folded into the rename step.
- `landskap` has 36 distinct raw values, but some are abbreviations/typos of a full name already
  present elsewhere in the column (e.g. `Bo` -> `Bohuslän`) - corrected in step 3 against
  `data/manual_resolutions/landskap_token_corrections.csv`.

Columns are renamed to their Swedish/SEAD-facing names in step 1 (`raa_id` -> `raa_nummer`,
`site_id` -> `lamningsnummer`, `lab_id` -> `lab_nummer`, `context_id` -> `feature_name`), then the
multi-valued `lamningsnummer`/`uppdragsnummer` cells are split into `_1`/`_2`/... columns in step 2
and `landskap` typos are corrected in step 3, before species and material get resolved and melted
in the steps that follow.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../..').resolve()))
from shared.resolution import common, species as species_resolution, material as material_resolution

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

MOD_DATASET_DIR = Path('../output/mod_dataset')

# next_available_dir never reuses a previous run's output folder - see shared/resolution/common.py.
# Every file this notebook writes lands in this one run-specific folder, so filenames below don't
# need their own version/suffix.
RUN_DIR = common.next_available_dir(MOD_DATASET_DIR, version='v1')
print(f'writing this run\'s outputs to {RUN_DIR}')


writing this run's outputs to ..\output\mod_dataset\v1_3


In [2]:
# .env lives at the true repo root, two levels up from current/notebooks/.
engine = common.get_db_engine('../../.env')

## 1. Load `StruckeC14_Sweden_v1.csv` and rename columns


In [3]:
RENAME_MAP = {
    'raa_id': 'raa_nummer',
    'site_id': 'lamningsnummer',
    'lab_id': 'lab_nummer',
    'context_id': 'feature_name',
}

df = pd.read_csv('../data/StruckeC14_Sweden_v1.csv', encoding='utf-8')
df = df.rename(columns=RENAME_MAP)
print(f'{len(df)} rows loaded, {len(df.columns)} columns')
df.columns.tolist()


30301 rows loaded, 37 columns


['socken',
 'landskap',
 'place_name',
 'raa_nummer',
 'site_type',
 'lamningsnummer',
 'uppdragsnummer',
 'lab_nummer',
 'c14_age_bp',
 'c14_error',
 'd13C',
 'pMC_value',
 'pMC_error',
 'c14_data_status',
 'comment',
 'assessed_relevant',
 'material',
 'species',
 'feature_name',
 'context_type',
 'northing_3006',
 'easting_3006',
 'longitude',
 'latitude',
 'location_precision',
 'cal_68_min',
 'cal_68_max',
 'cal_95_min',
 'cal_95_max',
 'median_cal_year',
 'calibration_method',
 'calibration_status',
 'author',
 'publication_year',
 'title',
 'journal',
 'place_of_publication']

## 2. Split multi-valued `lamningsnummer`/`uppdragsnummer` into extra columns

Some raw records list multiple site or project ids in a single cell, separated by commas (e.g.
`L2016:9874, L2015:343`). Split both `lamningsnummer` (renamed from `site_id` in step 1) and
`uppdragsnummer` on `,` into `{column}_1`, `{column}_2`, ... columns - row count stays the same,
only the column count grows. Records with a single value keep it in `{column}_1` with the rest
`NaN`; records with no value at all get `NaN` across the board.


In [4]:
def split_multivalue_column(frame, column):
    parts = frame[column].str.split(r'\s*,\s*', expand=True).apply(lambda col: col.str.strip())
    parts.columns = [f'{column}_{i + 1}' for i in range(parts.shape[1])]
    parts = parts.apply(lambda col: col.where(col.notna() & (col != ''), pd.NA))
    return frame.drop(columns=[column]).join(parts)

before_rows, before_cols = len(df), len(df.columns)
df = split_multivalue_column(df, 'lamningsnummer')
df = split_multivalue_column(df, 'uppdragsnummer')
print(f'{before_rows} rows unchanged ({len(df)} after) - {before_cols} columns -> {len(df.columns)} columns')
df.filter(regex=r'^(lamningsnummer|uppdragsnummer)_').head(10)


30301 rows unchanged (30301 after) - 37 columns -> 42 columns


,lamningsnummer_1,lamningsnummer_2,lamningsnummer_3,lamningsnummer_4,uppdragsnummer_1,uppdragsnummer_2,uppdragsnummer_3
0,L1969:5267,NaN,NaN,NaN,000103367,NaN,NaN
1,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
2,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
3,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
4,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
5,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
6,L1970:9431,NaN,NaN,NaN,NaN,NaN,NaN
7,L1970:9914,NaN,NaN,NaN,200600134,NaN,NaN
8,L1970:9914,NaN,NaN,NaN,200600134,NaN,NaN
9,L1970:9914,NaN,NaN,NaN,200600134,NaN,NaN


Sanity check: every real `lamningsnummer`/`uppdragsnummer` value follows a fixed-width format
(`Lyyyy:nnnnn`, up to 11 characters; a purely numeric project number, up to 9 characters), so a
split value longer than that would mean the `,` split above missed a comma. Assert the max
observed length stays within those bounds.


In [5]:
LAMNINGSNUMMER_MAX_LEN = 11
UPPDRAGSNUMMER_MAX_LEN = 9

lamningsnummer_cols = [c for c in df.columns if c.startswith('lamningsnummer_')]
uppdragsnummer_cols = [c for c in df.columns if c.startswith('uppdragsnummer_')]

lamningsnummer_max_len = df[lamningsnummer_cols].stack().str.len().max()
uppdragsnummer_max_len = df[uppdragsnummer_cols].stack().str.len().max()
print(f'max lamningsnummer_* value length: {lamningsnummer_max_len}')
print(f'max uppdragsnummer_* value length: {uppdragsnummer_max_len}')

assert lamningsnummer_max_len <= LAMNINGSNUMMER_MAX_LEN, (
    'a lamningsnummer_* value is longer than expected - the comma split above may have missed a comma'
)
assert uppdragsnummer_max_len <= UPPDRAGSNUMMER_MAX_LEN, (
    'an uppdragsnummer_* value is longer than expected - the comma split above may have missed a comma'
)


max lamningsnummer_* value length: 11.0
max uppdragsnummer_* value length: 9.0


## 3. Correct `landskap` values against the manual token-correction mapping

`data/manual_resolutions/landskap_token_corrections.csv` maps every one of the raw `landskap`
column's 36 distinct values to a corrected province name - most rows are already correct and map
to themselves, the rest are abbreviations/typos of a full name already present elsewhere in the
column (e.g. `Bo` -> `Bohuslän`, `Vg`/`vg` -> `Västergötland`, `Sk` -> `Skåne`). Unlike the species
mapping below, this one was built directly against this dataset revision's raw values, so a plain
join covers every row - no reconciliation/normalization pass needed.


In [6]:
LANDSKAP_MAPPING_PATH = Path('../data/manual_resolutions/landskap_token_corrections.csv')
landskap_map = pd.read_csv(
    LANDSKAP_MAPPING_PATH, keep_default_na=False, na_values=[''],
)[['landskap', 'manual_landskap']]

before_values = df['landskap'].nunique(dropna=False)
df = df.merge(landskap_map, on='landskap', how='left')

unmapped = sorted(df.loc[df['manual_landskap'].isna() & df['landskap'].notna(), 'landskap'].unique())
assert not unmapped, f'landskap values missing from the manual mapping: {unmapped}'

df = df.drop(columns=['landskap']).rename(columns={'manual_landskap': 'landskap'})
after_values = df['landskap'].nunique(dropna=False)
print(f'{before_values} raw landskap values -> {after_values} corrected landskap values')
df['landskap'].value_counts(dropna=False)


36 raw landskap values -> 25 corrected landskap values


landskap
Uppland          6045
Skåne            4798
Östergötland     3214
Södermanland     2957
Bohuslän         1946
Halland          1914
Småland          1799
Västergötland    1348
Närke            1332
Västmanland      1250
Dalarna           669
Blekinge          563
Värmland          379
Norrbotten        328
Gotland           294
Lappland          293
Medelpad          269
Öland             236
Jämtland          214
Ångermanland      116
Härjedalen        108
Hälsingland        88
Gästrikland        83
Västerbotten       42
Dalsland           16
Name: count, dtype: int64

## 4. Split/melt `species` into one value per row

Same approach as `archive/notebooks/species_study.ipynb`/`c14_dataset_tranformation.ipynb`:
lowercase, split on `,`/`/`, strip stray digits/`?`, melt so each record contributes one row per
split species value (records with no species value at all keep a single row with
`species_split = NaN`). Duplicated here rather than imported, since this raw-column-splitting step
is specific to how each dataset revision exports the `species` field.


In [7]:
import re

df['species'] = df['species'].str.lower()

noise_pattern = re.compile(r'[0-9?]')

def clean_text(value):
    if pd.isna(value) or value == '':
        return pd.NA
    cleaned = noise_pattern.sub('', value).strip()
    return cleaned if cleaned else pd.NA

species_parts = df['species'].str.split(r'[,/]', expand=True)
species_parts.columns = [f'species_{i + 1}' for i in range(species_parts.shape[1])]
species_parts = species_parts.apply(lambda col: col.map(clean_text))
df = df.join(species_parts)

species_cols = [c for c in df.columns if c.startswith('species_')]
species_split = df[species_cols].stack().dropna().droplevel(1).rename('species_split')
melted = df.drop(columns=species_cols).join(species_split).reset_index(drop=True)
print(f'{len(df)} original rows -> {len(melted)} rows after splitting/melting species')


30301 original rows -> 30805 rows after splitting/melting species


## 5. Reconcile `species_split` tokens against the manual token-correction mapping

`data/manual_resolutions/species_token_corrections.csv` (411 tokens) was built against the old
export's `species` text. Most of v1's tokens match it verbatim; where they don't, try normalizing
away the punctuation v1 added (`cf` -> `cf.`, trailing `.` after `sp`) before falling back to two
explicit overrides for the tokens that are genuinely new:

- `nakenkorn` -> `korn` (the old `naket korn`/`najet korn` rows were mapped to the uncertain
  `naket?` - v1's cleaned-up spelling is resolved directly to `korn` instead).
- `en möjl. gran` -> `gran`.

Anything left unresolved after that would need manual review before proceeding - asserted away
here since the reconciliation above accounts for every token in this revision.


In [8]:
MANUAL_MAPPING_PATH = Path('../data/manual_resolutions/species_token_corrections.csv')
manual_map_raw = pd.read_csv(
    MANUAL_MAPPING_PATH, keep_default_na=False, na_values=[''],
)[['species_split', 'manual_species']]

def normalize_token(t):
    if pd.isna(t):
        return t
    t = t.replace('cf.', 'cf').replace(' sp.', ' sp')
    return t.rstrip('.').strip()

known_tokens = set(manual_map_raw['species_split'].dropna().astype(str))
melted_tokens = set(melted['species_split'].dropna().astype(str))
unmatched = melted_tokens - known_tokens

norm_lookup = {}
for t in known_tokens:
    norm_lookup.setdefault(normalize_token(t), t)

EXPLICIT_OVERRIDES = {
    'nakenkorn': 'korn',
    'en möjl. gran': 'gran',
}

normalization_map = {}  # v1 token -> token it stands in for
for t in unmatched:
    if t in EXPLICIT_OVERRIDES:
        continue
    norm_t = normalize_token(t)
    if norm_t in norm_lookup:
        normalization_map[t] = norm_lookup[norm_t]

still_unresolved = sorted(unmatched - set(normalization_map) - set(EXPLICIT_OVERRIDES))
print(f'{len(unmatched)} tokens not present verbatim in the manual mapping')
print(f'  {len(normalization_map)} resolved via punctuation normalization')
print(f'  {len(EXPLICIT_OVERRIDES)} resolved via explicit override')
print(f'  {len(still_unresolved)} still unresolved: {still_unresolved}')
assert not still_unresolved, 'unexpected unmatched species_split tokens - needs manual review'


29 tokens not present verbatim in the manual mapping
  27 resolved via punctuation normalization
  2 resolved via explicit override
  0 still unresolved: []


In [9]:
def split_manual_species(value):
    if pd.isna(value):
        return [pd.NA]
    return [part.strip() for part in value.split(',')]

manual_species_split_map = pd.DataFrame([
    {'species_split': row.species_split, 'manual_species': part}
    for row in manual_map_raw.itertuples(index=False)
    for part in split_manual_species(row.manual_species)
])

override_rows = pd.DataFrame(
    [{'species_split': k, 'manual_species': v} for k, v in EXPLICIT_OVERRIDES.items()]
)
normalized_rows = pd.DataFrame([
    {'species_split': v1_token, 'manual_species': ms}
    for v1_token, base_token in normalization_map.items()
    for ms in split_manual_species(
        manual_map_raw.loc[manual_map_raw['species_split'] == base_token, 'manual_species'].iloc[0]
    )
])
manual_species_split_map_v1 = pd.concat(
    [manual_species_split_map, normalized_rows, override_rows], ignore_index=True
)

melted_with_manual = melted.merge(manual_species_split_map_v1, on='species_split', how='left')
print(f"{melted_with_manual['manual_species'].notna().sum()} of {len(melted_with_manual)} melted rows "
      f"matched a manual_species correction")
melted_with_manual[['species', 'species_split', 'manual_species']].head(10)


18746 of 30810 melted rows matched a manual_species correction


,species,species_split,manual_species
0,lind,lind,lind
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


## 6. Resolve SEAD taxonomy ids live and attach

`data/manual_resolutions/species_manual_resolution.csv` is the professionalized, hand-completed
species mapping (every row has `resolved_order`/`resolved_family`/`resolved_genus`/
`resolved_species`/`common_name_text` filled in). Unlike the old pipeline, this notebook does not
read a frozen `output/species/manual_species_resolved_with_ids_*.csv` - it calls
`shared.resolution.species.resolve_species_ids()` to recompute real `taxon_id`/`common_name_id`
values (and any newly-needed SEAD records) against the live DB every run, so the ids are always
current rather than pinned to whatever the DB looked like whenever someone last ran the old
notebook.


In [10]:
species_manual_df = pd.read_csv(
    '../data/manual_resolutions/species_manual_resolution.csv', keep_default_na=False, na_values=[''],
)
resolved_species, new_species_records = species_resolution.resolve_species_ids(species_manual_df, engine)

print(f"{resolved_species['taxon_id'].notna().sum()} of {len(resolved_species)} manual_species rows "
      f"resolved to a taxon_id ({int(resolved_species['taxon_id_is_new'].sum())} newly proposed)")
print(f'{len(new_species_records)} new SEAD records proposed for this list')


212 of 212 manual_species rows resolved to a taxon_id (139 newly proposed)
306 new SEAD records proposed for this list


In [11]:
SEAD_COLUMN_MAP = {
    'common_name_text': 'common_name',
    'resolved_species': 'sead_species',
    'resolved_genus': 'sead_genus',
    'resolved_family': 'sead_family',
    'resolved_order': 'sead_order',
    'taxon_id': 'sead_species_id',
    'resolved_genus_id': 'sead_genus_id',
    'resolved_family_id': 'sead_family_id',
    'resolved_order_id': 'sead_order_id',
    'common_name_id': 'sead_common_name_id',
    'common_name_language': 'sead_common_name_language_id',
}

resolved_for_join = resolved_species[['manual_species'] + list(SEAD_COLUMN_MAP)].rename(
    columns=SEAD_COLUMN_MAP
)

with_sead = melted_with_manual.merge(resolved_for_join, on='manual_species', how='left')
with_sead = with_sead.rename(columns={'manual_species': 'species_manually_assigned'})

print(f"{with_sead['sead_species_id'].notna().sum()} of {len(with_sead)} rows have a "
      f"sead_species_id attached")
with_sead[
    ['species', 'species_split', 'species_manually_assigned', 'common_name', 'sead_genus',
     'sead_family', 'sead_order', 'sead_species_id']
].head(10)


30810 of 30810 rows have a sead_species_id attached


,species,species_split,species_manually_assigned,common_name,sead_genus,sead_family,sead_order,sead_species_id
0,lind,lind,lind,lind,Tilia,Tiliaceae,Malvales,47020
1,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
2,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
3,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
4,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
5,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
6,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
7,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
8,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
9,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014


In [12]:
C14_WITH_SEAD_PATH = RUN_DIR / 'StruckeC14_Sweden_with_sead_taxonomy.csv'
with_sead.to_csv(C14_WITH_SEAD_PATH, index=False)
print(f'Saved {len(with_sead)} rows to {C14_WITH_SEAD_PATH}')

NEW_SPECIES_RECORDS_PATH = RUN_DIR / 'new_sead_records_species.csv'
new_species_records.to_csv(NEW_SPECIES_RECORDS_PATH, index=False)
print(f'Saved {len(new_species_records)} proposed new species/taxonomy records to {NEW_SPECIES_RECORDS_PATH}')


Saved 30810 rows to ..\output\mod_dataset\v1_3\StruckeC14_Sweden_with_sead_taxonomy.csv
Saved 306 proposed new species/taxonomy records to ..\output\mod_dataset\v1_3\new_sead_records_species.csv


## 7. Resolve material element ids live and attach

Same pattern as species: `data/manual_resolutions/material_manual_resolution.csv` is the
professionalized, hand-completed material breakdown. `shared.resolution.material.resolve_material_ids()`
recomputes real `tbl_abundance_elements`/`tbl_modification_types` ids live against the DB - no
`output/material/material_counts_resolved_with_ids_*.csv` dependency. v1's `material` text is
lowercase where the manual resolution was built against Title Case originals, so the join key is
lowercased on both sides (the display column itself is left untouched).


In [13]:
material_manual_df = pd.read_csv(
    '../data/manual_resolutions/material_manual_resolution.csv', keep_default_na=False, na_values=[''],
)
resolved_material, new_material_records = material_resolution.resolve_material_ids(material_manual_df, engine)

n_new_elements = sum(resolved_material[f'sead_element_{i}_is_new'].sum() for i in (1, 2, 3))
print(f'{int(n_new_elements)} element assignments are newly proposed, '
      f"{int(resolved_material['modification_type_is_new'].sum())} modification-type assignments are newly proposed")
print(f'{len(new_material_records)} new SEAD records proposed for this list')


37 element assignments are newly proposed, 0 modification-type assignments are newly proposed
20 new SEAD records proposed for this list


In [14]:
material_for_join = resolved_material[
    ['material', 'sead_element_1', 'sead_element_1_id', 'sead_element_2', 'sead_element_2_id',
     'sead_element_3', 'sead_element_3_id', 'sead_modification_type', 'sead_modification_type_id',
     'sead_record_type_id']
].copy()
material_for_join['material_lc'] = material_for_join['material'].str.lower().str.strip()
material_for_join = material_for_join.drop(columns=['material'])

with_sead['material_lc'] = with_sead['material'].str.lower().str.strip()
with_material = with_sead.merge(material_for_join, on='material_lc', how='left').drop(columns=['material_lc'])

print(f"{with_material['sead_element_1'].notna().sum()} of {len(with_material)} rows matched a "
      f"resolved material element")


30809 of 30810 rows matched a resolved material element


## 8. Melt into one material element per row

Same "stack the parts, drop the blanks" idea as the species split: `sead_element_1/2/3` become a
single `element_name` column, one row per element. Every output row now identifies exactly one
species and one material element - no measurement melt in this pipeline.


In [15]:
element_frames = []
for i in (1, 2, 3):
    other_cols = [f'sead_element_{j}{suffix}' for j in (1, 2, 3) if j != i for suffix in ('', '_id')]
    frame = with_material.drop(columns=other_cols).rename(
        columns={f'sead_element_{i}': 'element_name', f'sead_element_{i}_id': 'sead_element_id'}
    )
    element_frames.append(frame)

final_df = pd.concat(element_frames, ignore_index=True)
final_df = final_df[final_df['element_name'].notna()].reset_index(drop=True)

print(f'{len(with_material)} rows before the element melt -> {len(final_df)} rows after '
      f'(one row per species per material element)')
final_df[
    ['species_manually_assigned', 'material', 'element_name', 'sead_element_id',
     'sead_modification_type', 'sead_modification_type_id', 'sead_record_type_id']
].head(10)


30810 rows before the element melt -> 30819 rows after (one row per species per material element)


,species_manually_assigned,material,element_name,sead_element_id,sead_modification_type,sead_modification_type_id,sead_record_type_id
0,lind,träkol,Charcoal,50.0,NaN,NaN,9.0
1,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
2,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
3,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
4,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
5,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
6,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
7,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
8,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
9,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0


In [16]:
C14_FINAL_PATH = RUN_DIR / 'StruckeC14_Sweden_with_sead_taxonomy_and_material.csv'
final_df.to_csv(C14_FINAL_PATH, index=False)
print(f'Saved {len(final_df)} rows to {C14_FINAL_PATH}')

NEW_MATERIAL_RECORDS_PATH = RUN_DIR / 'new_sead_records_material.csv'
new_material_records.to_csv(NEW_MATERIAL_RECORDS_PATH, index=False)
print(f'Saved {len(new_material_records)} proposed new material records to {NEW_MATERIAL_RECORDS_PATH}')


Saved 30819 rows to ..\output\mod_dataset\v1_3\StruckeC14_Sweden_with_sead_taxonomy_and_material.csv
Saved 20 proposed new material records to ..\output\mod_dataset\v1_3\new_sead_records_material.csv
